# Lab 01-03 — Structure-preserving markdown chunking: split on the header hierarchy

**Track 01 · Chunking** — instead of chunking a markdown document by raw character counts, this lab splits it *by its header hierarchy*. `MarkdownHeaderTextSplitter` cuts the text at every `#` / `##` / `###` heading and carries the heading chain into each chunk's `metadata` (`{"H1": ..., "H2": ..., "H3": ...}`), so every chunk knows which section it belongs to.

This notebook is **self-contained**: it imports the LangChain markdown splitter directly — no repo component library. Every block of the pipeline is built right here: the raw markdown read, the header-aware splitter, and the metadata inspection all appear as plain code in the cells below, which is exactly how the shared components in `src/` work underneath.

```
raw markdown -> MarkdownHeaderTextSplitter(headers_to_split_on=[#/##/###]) -> chunks + header-chain metadata
Data  : Data/local-docs/docs/ (bge-embeddings.md, faiss-search.md)
        + the repo's own long markdown files (README.md, .omo/plans/layer1-rag-playbook.md)
```

Why this matters for retrieval: a plain character splitter produces chunks with no idea of their section, so a question about "persistence" can surface a chunk from the wrong part of the doc. With header metadata, retrieval can scope by section — e.g. filter to chunks whose `H1 == "FAISS similarity search"` before ranking, or show the section path alongside every hit.

The primary demo is the repo's own long markdown files — `README.md` and `.omo/plans/layer1-rag-playbook.md` — which have a real `#`/`##`/`###` hierarchy worth splitting on. The tiny docs under `Data/local-docs/` stay as a second, side-by-side demo (their chunk indices 0-3 are stable, so the strip_headers comparison and section-scoped retrieval blocks keep using them).

Compare against `RecursiveCharacterTextSplitter` (lab 01) and `TokenTextSplitter` (lab 02): those split on size, this one splits on structure.


## Setup

Two prerequisites must hold before this notebook will run:

- **local-docs samples on disk** — `Data/local-docs/docs/` with `bge-embeddings.md` and `faiss-search.md` (already fetched by the repo's manifest-verified fetchers);
- **the repo's own markdown files** — `README.md` and `.omo/plans/layer1-rag-playbook.md` at the repo root (they are this lab's primary demo documents).

No repo imports are needed: everything this notebook uses comes from `langchain-core` and `langchain-text-splitters`. The imports cell walks up to the repo root and cd's into it, because a notebook has no `__file__` — so every `Data/...` path resolves exactly like the lab script. Unlike the Curriculum notebook, there is no `sys.path` trick: nothing is imported from `src/`.

The next cell installs the notebook-specific dependencies (a no-op if you already ran `pip install -r requirements.txt`).


In [ ]:
# Lab-specific dependencies (already in requirements.txt — the install
# below is a no-op if you have run `pip install -r requirements.txt`).
%pip install -q langchain-core langchain-text-splitters


In [ ]:
# Bootstrap: stdlib imports + repo-root walk (no sys.path tricks).
from __future__ import annotations

import os
from pathlib import Path

# LangChain markdown splitter — the only library this notebook needs.
# Nothing is imported from the repo's src/ component library.
from langchain_core.documents import Document  # noqa: E402
from langchain_text_splitters import MarkdownHeaderTextSplitter  # noqa: E402

# A notebook has no __file__, so walk up from the cwd to the repo root and
# cd into it — Data/... paths then resolve exactly like the lab script.
REPO_ROOT = Path.cwd()
for _candidate in [Path.cwd(), *Path.cwd().parents]:
    if (_candidate / "src" / "curriculum").is_dir() and (_candidate / "NoteBooks").is_dir():
        REPO_ROOT = _candidate
        break
os.chdir(REPO_ROOT)


## 1. Configuration

`HEADERS_TO_SPLIT_ON` is the heading chain the splitter cuts on — the module-level constant that makes the lab tweakable: add `("####", "H4")` to split deeper, or drop `("###", "H3")` to merge H3 sections into their H2 parent. `DOCS_DIR`/`DOC_FILES` are the small local-docs samples (stable chunk indices 0-3); `LONG_MD_PATHS` are the repo's own long markdown files — the primary demo documents with a real `#`/`##`/`###` hierarchy.


In [ ]:
# --------------------------------------------------------------------------
# 1. Configuration — tweak these to rerun the experiment
# --------------------------------------------------------------------------
HEADERS_TO_SPLIT_ON: list[tuple[str, str]] = [
    ("#", "H1"),
    ("##", "H2"),
    ("###", "H3"),
]

DOCS_DIR = Path("Data/local-docs/docs")
DOC_FILES = ["bge-embeddings.md", "faiss-search.md"]

# The repo's own long markdown files (paths relative to the repo root) — the
# primary demo documents with a real #/##/### hierarchy.
LONG_MD_PATHS = ["README.md", ".omo/plans/layer1-rag-playbook.md"]


## 2. Load + split — markdown read inline, split on the header hierarchy

`load_markdown` reads a markdown file as raw text (no loader needed for plain `.md`). `split_markdown` builds a `MarkdownHeaderTextSplitter` on the configured chain and returns the section chunks; with `strip_headers=False` (the default) the heading line stays inside the chunk, so the chunk is self-contained when read or embedded; with `strip_headers=True` the heading lives only in metadata. `distinct_h1_values` collects the distinct H1 values across a chunk list in first-appearance order, and `preview` truncates content for printing.


In [ ]:
# --------------------------------------------------------------------------
# 2. Load + split — markdown read inline, split on the header hierarchy
# --------------------------------------------------------------------------
def load_markdown(path: Path) -> str:
    """Read a markdown file as raw text (no loader needed for plain .md)."""
    return path.read_text(encoding="utf-8")


def distinct_h1_values(chunks: list[Document]) -> list[str]:
    """Distinct H1 values across chunks, in first-appearance order."""
    seen: list[str] = []
    for chunk in chunks:
        h1 = chunk.metadata.get("H1")
        if h1 is not None and h1 not in seen:
            seen.append(h1)
    return seen


def split_markdown(text: str, strip_headers: bool) -> list[Document]:
    """Split markdown text on its header hierarchy.

    Args:
        text: raw markdown content.
        strip_headers: if True the heading line is removed from the chunk
            content and kept only in metadata; if False the heading stays in
            the content too.

    Returns:
        List of Documents whose ``metadata`` carries the header chain
        (e.g. ``{"H1": "FAISS similarity search", "H2": "Persistence"}``).
    """
    splitter = MarkdownHeaderTextSplitter(
        headers_to_split_on=HEADERS_TO_SPLIT_ON,
        strip_headers=strip_headers,
    )
    return splitter.split_text(text)


def preview(text: str, limit: int = 200) -> str:
    """Truncate a chunk's content for printing."""
    return text[:limit] + ("..." if len(text) > limit else "")


## 3. Experiment — one splitter config, two document sets

`run_experiment` walks the whole demo. Setup prints the splitter config and both document sets. Then the local docs are split with `strip_headers=False` (headers kept in content) and the chunk counts reported, with the first chunk's header-chain metadata shown. Then the real demo: the repo's own long markdown files, split the same way, reporting chunk counts and distinct H1 values per file. Then the side-by-side — the same `## Persistence` section of `faiss-search.md` split both ways (`strip_headers=False` vs `True`) — and finally the payoff: because every chunk carries its H1/H2/H3 chain, a vector store can filter before ranking, which the section-scoped retrieval block demonstrates as a fictional metadata-filter query.


In [ ]:
# --------------------------------------------------------------------------
# 3. Experiment — one splitter config, two document sets
# --------------------------------------------------------------------------
def run_experiment() -> dict:
    # --- load: raw markdown text from disk --------------------------------
    docs: dict[str, str] = {
        name: load_markdown(DOCS_DIR / name) for name in DOC_FILES
    }

    # --- split: structure-preserving, headers kept in content -------------
    # strip_headers=False (the default) keeps the heading line inside the
    # chunk, so the chunk is self-contained when read or embedded.
    chunks_kept: dict[str, list[Document]] = {
        name: split_markdown(text, strip_headers=False)
        for name, text in docs.items()
    }
    total_kept = sum(len(c) for c in chunks_kept.values())
    first_local = chunks_kept[DOC_FILES[0]][0]

    # --- the real demo: the repo's own long markdown docs -----------------
    # README.md and the playbook plan are long, real markdown files with a
    # genuine #/##/### hierarchy — exactly what this splitter is for.
    long_chunks: dict[str, list[Document]] = {
        path: split_markdown(load_markdown(Path(path)), strip_headers=False)
        for path in LONG_MD_PATHS
    }
    total_long = sum(len(c) for c in long_chunks.values())
    long_h1s: dict[str, list[str]] = {
        path: distinct_h1_values(chunks) for path, chunks in long_chunks.items()
    }
    readme_first = long_chunks["README.md"][0]

    # --- side by side: strip_headers=False vs True for one section ---------
    # The same H2 section of faiss-search.md, split both ways.
    faiss_text = docs["faiss-search.md"]
    kept = split_markdown(faiss_text, strip_headers=False)
    stripped = split_markdown(faiss_text, strip_headers=True)
    # chunk 0 = "# FAISS similarity search" intro, chunk 1 = "## What it
    # does", chunk 2 = "## Persistence", chunk 3 = "## Querying".

    return {
        "docs": docs,
        "chunks_kept": chunks_kept,
        "total_kept": total_kept,
        "first_local": first_local,
        "long_chunks": long_chunks,
        "total_long": total_long,
        "long_h1s": long_h1s,
        "readme_first": readme_first,
        "kept": kept,
        "stripped": stripped,
    }


## 4. Demo — print the artifact

`print_demo(exp)` prints the artifact in the lab's order: the splitter config; the local-docs chunk counts and the first chunk's header-chain metadata; the long repo docs with their chunk counts and distinct H1 values, plus the README's first chunk; the `## Persistence` side-by-side (heading kept in content vs heading only in metadata); the section-scoped retrieval demo; and the takeaway.


In [ ]:
# --------------------------------------------------------------------------
# 4. Demo — print the artifact
# --------------------------------------------------------------------------
def print_demo(exp: dict) -> None:
    chunks_kept = exp["chunks_kept"]
    long_chunks = exp["long_chunks"]
    long_h1s = exp["long_h1s"]
    kept = exp["kept"]
    stripped = exp["stripped"]

    # --- setup: one splitter config, two document sets -------------------
    print(f"Headers to split on: {HEADERS_TO_SPLIT_ON}")
    print(f"Local docs: {DOC_FILES}")
    print(f"Long repo docs: {LONG_MD_PATHS}\n")

    print(f"strip_headers=False -> {exp['total_kept']} chunk(s) across {len(chunks_kept)} file(s)")
    for name, chunks in chunks_kept.items():
        print(f"  {name}: {len(chunks)} chunk(s)")

    # --- inspect metadata: the header chain each chunk carries -------------
    first = exp["first_local"]
    print(f"\nFirst chunk of {DOC_FILES[0]} — metadata (header chain):")
    print(f"  {first.metadata}")
    print(f"  Content preview: {preview(first.page_content)!r}")

    print(f"\nLong repo markdown -> {exp['total_long']} chunk(s) across {len(LONG_MD_PATHS)} file(s)")
    for path, chunks in long_chunks.items():
        h1s = long_h1s[path]
        print(f"  {path}: {len(chunks)} chunk(s), {len(h1s)} distinct H1 value(s)")

    readme_first = exp["readme_first"]
    print(f"\nFirst chunk of README.md — metadata (header chain):")
    print(f"  {readme_first.metadata}")
    print(f"  Content preview: {preview(readme_first.page_content)!r}")

    # --- side by side: strip_headers=False vs True for one section ---------
    print("\nSide by side — '## Persistence' section of faiss-search.md:")
    print("  strip_headers=False (heading inside content):")
    print(f"    metadata: {kept[2].metadata}")
    print(f"    content : {preview(kept[2].page_content)!r}")
    print("  strip_headers=True  (heading only in metadata):")
    print(f"    metadata: {stripped[2].metadata}")
    print(f"    content : {preview(stripped[2].page_content)!r}")

    # --- the payoff: section-scoped retrieval ------------------------------
    # Because every chunk carries its H1/H2/H3 chain, a vector store can
    # filter before ranking — a fictional metadata-filter query:
    print("\nSection-scoped retrieval (fictional metadata filter):")
    print("  query = 'How do I persist a FAISS index?'")
    print("  filter = {'H1': 'FAISS similarity search'}  # scope to one doc")
    print("  -> only chunks whose metadata['H1'] matches are embedded/ranked,")
    print("     so a 'persistence' hit can never come from the BGE page.")
    print("\nTakeaway: structure-preserving splitting carries the section")
    print("hierarchy into chunk metadata — retrieval can scope by section.")


## 5. Verification gate

`verify_gate(exp)` enforces the lab's hard checks: both local docs split into their stable 4 section chunks each; the first chunk of `bge-embeddings.md` carries the `H1` header chain; the `## Persistence` section of `faiss-search.md` is chunk index 2 in BOTH split modes with the identical header chain, and the heading text is inside the kept chunk's content but not in the stripped chunk's content; and both long repo docs produce chunks with at least one distinct H1. Every check should print PASS.


In [ ]:
# --------------------------------------------------------------------------
# 5. Verification gate
# --------------------------------------------------------------------------
def verify_gate(exp: dict) -> int:
    checks: list[tuple[str, bool]] = []
    chunks_kept = exp["chunks_kept"]
    kept = exp["kept"]
    stripped = exp["stripped"]
    long_chunks = exp["long_chunks"]
    long_h1s = exp["long_h1s"]

    for name in DOC_FILES:
        checks.append((
            f"{name} split into its section chunks",
            len(chunks_kept[name]) == 4,
        ))
    checks.append((
        f"first chunk of {DOC_FILES[0]} carries the header chain",
        exp["first_local"].metadata == {"H1": "BGE embeddings"},
    ))
    checks.append((
        "'## Persistence' section is chunk index 2 (strip_headers=False)",
        kept[2].metadata == {"H1": "FAISS similarity search", "H2": "Persistence"},
    ))
    checks.append((
        "'## Persistence' section is chunk index 2 (strip_headers=True)",
        stripped[2].metadata == {"H1": "FAISS similarity search", "H2": "Persistence"},
    ))
    checks.append((
        "heading kept inside content (strip_headers=False)",
        "## Persistence" in kept[2].page_content,
    ))
    checks.append((
        "heading only in metadata (strip_headers=True)",
        "## Persistence" not in stripped[2].page_content,
    ))
    for path in LONG_MD_PATHS:
        checks.append((
            f"long repo doc splits into chunks: {path}",
            len(long_chunks[path]) > 0,
        ))
        checks.append((
            f"long repo doc carries H1 metadata: {path}",
            len(long_h1s[path]) >= 1,
        ))

    print("verification gate:")
    for label, ok in checks:
        print(f"  [{'PASS' if ok else 'FAIL'}] {label}")
    return 0 if all(ok for _, ok in checks) else 1


## Run the experiment

A few seconds: four markdown file reads and header-hierarchy splits — no downloads, no API calls, no embeddings. `exp` holds everything the demo and gate need.


In [ ]:
exp = run_experiment()


### Demo — the artifact

The header chain each chunk carries, from the tiny local docs to the repo's own long markdown files: chunk counts, distinct H1 values, the `## Persistence` section split both ways, and the section-scoped retrieval payoff.


In [ ]:
print_demo(exp)


### Verification gate

Expect every check to PASS — the same gate the CI-style `--verify` run enforces. If any line shows FAIL, check the local-docs samples and the repo markdown files are intact.


In [ ]:
verify_gate(exp)
